In [15]:
!pip install -U huggingface_hub datasets transformers evaluate seqeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 95.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [7]:
#create dataset
data = [
    {"tokens": ["John", "works", "at", "Google"], "labels": ["NOUN", "VERB", "ADP", "NOUN"]},
    {"tokens": ["She", "is", "reading", "a", "book"], "labels": ["PRON", "VERB", "VERB", "DET", "NOUN"]},
    {"tokens": ["They", "play", "football"], "labels": ["PRON", "VERB", "NOUN"]},
    {"tokens": ["I", "love", "Python"], "labels": ["PRON", "VERB", "NOUN"]},
]

In [8]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset = dataset.train_test_split(test_size=0.2)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 3
    })
    test: Dataset({
        features: ['tokens', 'labels'],
        num_rows: 1
    })
})


In [9]:
#create label mapping
label_list = ["NOUN", "VERB", "ADP", "PRON", "DET"]

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

In [10]:
#load tokenization
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [11]:
#tokenization + label allignment
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    word_ids = tokenized_inputs.word_ids()

    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            labels.append(-100)
        elif word_idx != previous_word_idx:
            labels.append(label2id[example["labels"][word_idx]])
        else:
            labels.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [12]:
tokenized_dataset = dataset.map(tokenize_and_align_labels)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [13]:
#model setup
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized be

In [18]:
#import transformers
from transformers import DataCollatorForTokenClassification,TrainingArguments, Trainer

#create collator
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator
)


In [19]:
#load metric
import evaluate
import numpy as np

metric = evaluate.load("seqeval")

In [21]:
#define evaluation function
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [id2label[p] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [id2label[l] for p, l in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"]
    }

In [24]:
#attach & evaluate
trainer.compute_metrics = compute_metrics

predictions = trainer.predict(tokenized_dataset["test"])

print(predictions.metrics)

{'test_loss': 1.4440217018127441, 'test_model_preparation_time': 0.0054, 'test_precision': 0.3333333333333333, 'test_recall': 0.25, 'test_f1': 0.28571428571428575, 'test_runtime': 0.0535, 'test_samples_per_second': 18.693, 'test_steps_per_second': 18.693}


In [28]:
import torch

# set device
device = "cuda" if torch.cuda.is_available() else "cpu"

# move model to device
model.to(device)

# sentence
sentence = "John works at Google in California"
tokens = sentence.split()

# tokenize
inputs = tokenizer(tokens, is_split_into_words=True, return_tensors="pt")

# move inputs to same device
inputs = {k: v.to(device) for k, v in inputs.items()}

# prediction
outputs = model(**inputs).logits
predictions = outputs.argmax(dim=2)

# convert to labels
predicted_labels = [id2label[p.item()] for p in predictions[0]]

# print output
for token, label in zip(tokens, predicted_labels):
    print(token, "→", label)

John → PRON
works → DET
at → PRON
Google → ADP
in → NOUN
California → ADP


## Comparison between POS Tagging and Chunking

POS Tagging assigns grammatical categories to each word such as noun, verb, adjective, etc. It focuses on individual words and is relatively simpler.

Chunking, on the other hand, groups words into meaningful phrases such as noun phrases or verb phrases. It considers relationships between words and is more complex than POS tagging.

Therefore, POS tagging is a word-level task, whereas chunking is a phrase-level task.

## Report

### Differences between POS Tagging and Chunking
POS tagging labels each word with its grammatical category, while chunking groups words into phrases like noun phrases or verb phrases.

### Challenges Faced
- Dataset compatibility issues with latest Hugging Face libraries
- Handling subword tokenization
- Aligning labels using -100 for special tokens
- Padding sequences using data collator

### Observations and Insights
- BERT performs effectively for sequence labeling tasks
- POS tagging is easier compared to chunking
- Model performance improves with more training data